# 5장 논문의 KM 곡선에서 환자 자료 되살리기 (R과 통계분석)

논문의 Kaplan-Meier 그림에서 읽어 낸 좌표와 위험집합 표로 환자 단위 자료를 되살리고(Guyot 방법, IPDfromKM 패키지), 원자료와 비교합니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 그림에서 읽어 낸 것 준비하기

### 셀 1. 그림에서 읽어 낸 좌표

In [ ]:
options(HTTPUserAgent = "Mozilla/5.0")
BASE <- "https://socialp-ajou.tecentriq12.workers.dev/data/"
dig <- read.csv(paste0(BASE, "rcc_km_digitized.csv"))
dim(dig)
head(dig, 8)
c(time_reversed = sum(diff(dig$time) < 0), surv_up = sum(diff(dig$surv) > 0))

### 셀 2. 읽어 낸 좌표 그려 보기

In [ ]:
plot(dig$time, dig$surv, pch = 20, cex = 0.5, ylim = c(0, 1), xaxt = "n",
     xlab = "Months", ylab = "Overall survival (read from figure)")
axis(1, at = seq(0, 30, by = 6))

### 셀 3. 위험집합 표와 총 사건 수 옮겨 적기

In [ ]:
trisk <- c(0, 6, 12, 18, 24)            # 그림 아래 위험집합 표의 시점(개월)
nrisk <- c(300, 265, 238, 202, 167)     # 그 시점에 위험집합에 남은 환자 수
tot_events <- 145                       # 본문에 적힌 총 사망 수

## b. 환자 자료 되살리기

### 셀 4. preprocess()로 좌표 정리하기

In [ ]:
install.packages("IPDfromKM")
library(IPDfromKM)
pre <- preprocess(dat = dig, trisk = trisk, nrisk = nrisk,
                  totalpts = 300, maxy = 1)
nrow(pre$preprocessdat)
head(pre$preprocessdat, 4)
pre$intervalIndex

### 셀 5. getIPD()로 환자 자료 되살리기

In [ ]:
ipd <- getIPD(prep = pre, armID = 1, tot.events = tot_events)
summary(ipd)

### 셀 6. 되살린 자료 살펴보기

In [ ]:
rec <- ipd$IPD
head(rec)
table(rec$status)
library(survival)
km_r <- survfit(Surv(time, status) ~ 1, data = rec)
summary(km_r, times = trisk)[c("n.risk", "surv")]

## c. 원자료와 비교하기

### 셀 7. 원자료와 생존율 비교

In [ ]:
orig <- read.csv(paste0(BASE, "rcc_trial.csv"))     # 정답 확인용 원자료
km_o <- survfit(Surv(months, event) ~ 1, data = orig)
at <- c(6, 12, 18, 24, 29)
round(cbind(months = at, original = summary(km_o, times = at)$surv,
            reconstructed = summary(km_r, times = at)$surv), 4)
after24 <- function(t, e)            # 24개월 뒤의 사망 수와 위험집합에 머문 기간(사람-월)
  c(deaths = sum(e[t > 24]), person_months = round(sum(pmax(0, t - 24)), 1))
rbind(original = after24(orig$months, orig$event), reconstructed = after24(rec$time, rec$status))

### 셀 8. 두 곡선 겹쳐 그리기

In [ ]:
plot(km_o, conf.int = FALSE, xlab = "Months", ylab = "Overall survival", xaxt = "n")
lines(km_r, conf.int = FALSE, col = "#c8553d", lty = 2)
points(dig$time, dig$surv, pch = 20, cex = 0.3, col = "gray50")
axis(1, at = seq(0, 30, by = 6))
legend("bottomleft", c("Original data", "Reconstructed", "Points read"),
       col = c("black", "#c8553d", "gray50"), lty = c(1, 2, NA),
       pch = c(NA, NA, 20), bty = "n")

### 셀 9. 여섯 분포로 외삽한 결과 비교

In [ ]:
library(flexsurv)
dists <- c(exp = "exp", weibull = "weibull", gompertz = "gompertz",
           lnorm = "lnorm", llogis = "llogis", gengamma = "gengamma")
ext_tab <- function(formula, data) {          # 여섯 분포의 AIC 차이, 5년 생존율, 20년 평균 생존
  fits <- lapply(dists, function(k) flexsurvreg(formula, data = data, dist = k))
  aic  <- sapply(fits, AIC)
  data.frame(dAIC = round(aic - min(aic), 1),
             S5y  = round(100 * sapply(fits, function(f)
                      summary(f, t = 60, ci = FALSE, tidy = TRUE)$est), 1),
             RMST20 = round(sapply(fits, function(f)
                      summary(f, t = 240, type = "rmst", ci = FALSE, tidy = TRUE)$est) / 12, 2))
}
cbind(ext_tab(Surv(months, event) ~ 1, orig), ext_tab(Surv(time, status) ~ 1, rec))

## d. 위험집합 표가 촘촘하면

### 셀 10. 3개월 간격의 위험집합 표로 다시

In [ ]:
trisk3 <- seq(0, 27, by = 3)
nrisk3 <- c(300, 283, 265, 251, 238, 223, 202, 185, 167, 80)
rec3 <- getIPD(preprocess(dig, trisk = trisk3, nrisk = nrisk3, totalpts = 300, maxy = 1),
               armID = 1, tot.events = tot_events)$IPD
c(events = sum(rec3$status), after24 = sum(rec3$status[rec3$time > 24]))
ext_tab(Surv(time, status) ~ 1, rec3)

## e. 과제 정답

### 셀 11. 과제 1 정답: 위험집합 표가 없으면

In [ ]:
rec0 <- getIPD(preprocess(dig, trisk = 0, nrisk = 300, totalpts = 300, maxy = 1),
               armID = 1, tot.events = tot_events)$IPD
c(events = sum(rec0$status))
summary(survfit(Surv(time, status) ~ 1, data = rec0), times = trisk)[c("n.risk", "surv")]

### 셀 12. 과제 2 정답: 읽기 오차가 없는 좌표로

In [ ]:
step <- data.frame(time = c(0, rep(km_o$time[km_o$n.event > 0], each = 2)),
                   surv = c(1, as.vector(rbind(c(1, head(km_o$surv[km_o$n.event > 0], -1)),
                                               km_o$surv[km_o$n.event > 0]))))
step <- rbind(step, data.frame(time = max(orig$months), surv = tail(km_o$surv, 1)))
rec_x <- getIPD(preprocess(step, trisk = trisk, nrisk = nrisk, totalpts = 300, maxy = 1),
                armID = 1, tot.events = tot_events)$IPD
c(events = sum(rec_x$status), after24 = sum(rec_x$status[rec_x$time > 24]))

### 셀 13. 과제 3 정답: 세로축이 백분율이면

In [ ]:
dig_pct <- transform(dig, surv = surv * 100)      # 세로축이 0-100%인 그림
rec_p <- getIPD(preprocess(dig_pct, trisk = trisk, nrisk = nrisk, totalpts = 300, maxy = 100),
                armID = 1, tot.events = tot_events)$IPD
all.equal(rec_p, rec)